# SVIES — YOLOv8 Model Training Pipeline
## NVIDIA Tesla V100-SXM2 (32GB VRAM) | CUDA 12.5
### Smart Vehicle Intelligence & Enforcement System
**Models:** Plate Detection | Helmet Detection | Indian Vehicle Classification
**Architecture:** YOLOv8s (Small, 11.2M params) — optimal for V100

In [ ]:
# ══════════════════════════════════════════════════════════
# 1. Environment Setup
# ══════════════════════════════════════════════════════════
!pip3 install ultralytics roboflow torch torchvision --quiet

import torch
import os
from pathlib import Path
from datetime import datetime

# Verify GPU
print(f"PyTorch: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    print(f"CUDA Version: {torch.version.cuda}")

# Project paths
PROJECT_DIR = Path("./svies_training")
PROJECT_DIR.mkdir(exist_ok=True)
MODELS_DIR = PROJECT_DIR / "models"
MODELS_DIR.mkdir(exist_ok=True)
DATASETS_DIR = PROJECT_DIR / "datasets"
DATASETS_DIR.mkdir(exist_ok=True)

print(f"\nTraining directory: {PROJECT_DIR.resolve()}")
print(f"Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

## 2. Dataset Download from Roboflow
We use **5 datasets** (merged into 3 training sets) from Roboflow Universe:

### Model 1 — Indian License Plate Detector (merged ~2,100 images)
| Dataset | Images | Class |
|---------|--------|-------|
| `indian-number-plate` v3 | 1,683 | IndianNumberPlate |
| `indian-cars-number-plate` v1 | 418 | License-Plate |

### Model 2 — Helmet Violation Detector (merged ~2,520 images)
| Dataset | Images | Classes |
|---------|--------|---------|
| `motorcycle-helmet-detection` v2 | 2,215 | no helmet, full-faced, half-faced, invalid |
| `two-wheeler-helmet` v1 | 305 | with_helmet, without_helmet |

*Remapped to binary:* `with_helmet` (full-faced + half-faced) / `without_helmet` (no helmet). `invalid` dropped.

### Model 3 — Vehicle Type Detector (627 images)
| Dataset | Images | Classes |
|---------|--------|---------|
| `vehicles-openimages` v1 | 627 | Car, Bus, Motorcycle, Truck, Ambulance |

In [ ]:
# v4 FINAL FIX: After regeneration, wait then force download (skip link test)
import shutil, subprocess, json, time
import requests as http_req

ROBOFLOW_API_KEY = os.environ.get("ROBOFLOW_API_KEY", "YOUR_ROBOFLOW_API_KEY_HERE")

def find_link(data):
    if isinstance(data, dict):
        for k, v in data.items():
            if k == "link" and isinstance(v, str) and v.startswith("http"):
                return v
            f = find_link(v)
            if f: return f
    elif isinstance(data, list):
        for i in data:
            f = find_link(i)
            if f: return f
    return None

def download_dataset(ws, slug, ver, fmt, loc):
    if os.path.exists(loc): shutil.rmtree(loc)
    os.makedirs(loc, exist_ok=True)
    
    url = f"https://api.roboflow.com/{ws}/{slug}/{ver}/{fmt}"
    
    # Always force regenerate — old exports are all expired
    print(f"  Regenerating export...")
    http_req.post(url, params={"api_key": ROBOFLOW_API_KEY})
    
    # Wait for progress to hit 1 (complete)
    link = None
    for i in range(60):
        time.sleep(5)
        r = http_req.get(url, params={"api_key": ROBOFLOW_API_KEY})
        data = r.json()
        link = find_link(data)
        progress = data.get("progress", "?")
        print(f"    progress={progress} ({(i+1)*5}s)")
        if progress == 1 and link:
            break
    
    if not link:
        raise RuntimeError(f"No link for {slug}")
    
    # KEY FIX: Wait 60s extra for GCS CDN to propagate the new file
    print(f"  Export done! Waiting 60s for CDN propagation...")
    time.sleep(60)
    
    # Re-fetch link (might be a new URL now)
    r = http_req.get(url, params={"api_key": ROBOFLOW_API_KEY})
    fresh_link = find_link(r.json())
    if fresh_link:
        link = fresh_link
    
    # Download
    dl = os.path.join(loc, "archive.zip")
    print(f"  Downloading...")
    os.system(f'curl -sL --insecure -o "{dl}" "{link}"')
    
    fsize = os.path.getsize(dl)
    print(f"  Size: {fsize/1e6:.1f} MB")
    
    if fsize < 1000:
        with open(dl, "r", errors="ignore") as f:
            print(f"  ERROR: {f.read(300)}")
        raise RuntimeError("Still expired")
    
    ftype = subprocess.check_output(["file","-b",dl]).decode().strip()
    print(f"  Type: {ftype}")
    
    if "Zip" in ftype:
        os.system(f'unzip -q -o "{dl}" -d "{loc}"')
    elif "gzip" in ftype:
        os.system(f'tar xzf "{dl}" -C "{loc}"')
    else:
        os.system(f'unzip -q -o "{dl}" -d "{loc}" 2>/dev/null')
    
    os.remove(dl)
    total = sum(1 for _ in Path(loc).rglob("*") if _.is_file())
    print(f"  ✓ {total} files extracted\n")

WS = "roboflow-universe-projects"

for name, slug, ver, dest in [
    ("Indian Number Plate",          "indian-number-plate",         3, "plates_primary"),
    ("Indian Cars Number Plate",     "indian-cars-number-plate",    1, "plates_supplement"),
    ("Motorcycle Helmet Detection",  "motorcycle-helmet-detection", 2, "helmet_primary"),
    ("Two Wheeler Helmet",           "two-wheeler-helmet",          1, "helmet_supplement"),
    ("Vehicles OpenImages",          "vehicles-openimages",         1, "vehicles"),
]:
    print(f"{'='*55}\n{name}\n{'='*55}")
    download_dataset(WS, slug, ver, "yolov8", str(DATASETS_DIR / dest))

print("✓ All 5 datasets downloaded!")

## 2B. Merge Datasets & Remap Classes
- **Plates**: Merge two single-class plate datasets into one (~2,100 images)
- **Helmets**: Remap `motorcycle-helmet-detection` classes to binary (`with_helmet` / `without_helmet`), drop `invalid`, then merge with `two-wheeler-helmet`
- **Vehicles**: No merge needed (single dataset)

In [ ]:
# ══════════════════════════════════════════════════════════
# 2B. Merge Datasets & Remap Classes
# ══════════════════════════════════════════════════════════
import yaml

def copy_split(src_dir, dst_dir, prefix, class_remap=None, drop_classes=None):
    """Copy images+labels from src to dst, optionally remapping YOLO class IDs."""
    count = 0
    for split in ["train", "valid", "test"]:
        src_imgs = src_dir / split / "images"
        src_lbls = src_dir / split / "labels"
        dst_imgs = dst_dir / split / "images"
        dst_lbls = dst_dir / split / "labels"
        dst_imgs.mkdir(parents=True, exist_ok=True)
        dst_lbls.mkdir(parents=True, exist_ok=True)

        if not src_imgs.exists():
            continue

        for img in src_imgs.iterdir():
            if img.suffix.lower() not in (".jpg", ".jpeg", ".png", ".bmp"):
                continue
            new_name = f"{prefix}_{img.name}"
            shutil.copy2(img, dst_imgs / new_name)

            lbl = src_lbls / f"{img.stem}.txt"
            dst_lbl = dst_lbls / f"{prefix}_{img.stem}.txt"
            if lbl.exists():
                if class_remap or drop_classes:
                    lines = []
                    for line in lbl.read_text().strip().split("\n"):
                        if not line.strip():
                            continue
                        parts = line.strip().split()
                        cls_id = int(parts[0])
                        if drop_classes and cls_id in drop_classes:
                            continue
                        if class_remap and cls_id in class_remap:
                            parts[0] = str(class_remap[cls_id])
                        lines.append(" ".join(parts))
                    if lines:
                        dst_lbl.write_text("\n".join(lines) + "\n")
                else:
                    shutil.copy2(lbl, dst_lbl)
            count += 1
    return count

def write_data_yaml(dst_dir, class_names):
    data = {
        "train": str((dst_dir / "train" / "images").resolve()),
        "val": str((dst_dir / "valid" / "images").resolve()),
        "test": str((dst_dir / "test" / "images").resolve()),
        "nc": len(class_names),
        "names": class_names,
    }
    yaml_path = dst_dir / "data.yaml"
    with open(yaml_path, "w") as f:
        yaml.dump(data, f, default_flow_style=False)
    return yaml_path

def load_class_names(dataset_dir):
    """Read class names from a Roboflow data.yaml."""
    yaml_path = dataset_dir / "data.yaml"
    with open(yaml_path) as f:
        d = yaml.safe_load(f)
    return d.get("names", [])

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# MERGE 1: Indian License Plates
# Both datasets are single-class (class 0 = plate) → direct merge
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print("=" * 55)
print("MERGING: Indian License Plate Datasets")
print("=" * 55)

plates_merged = DATASETS_DIR / "plates_merged"
if plates_merged.exists():
    shutil.rmtree(plates_merged)

n1 = copy_split(DATASETS_DIR / "plates_primary", plates_merged, "inp")
n2 = copy_split(DATASETS_DIR / "plates_supplement", plates_merged, "icnp")
plates_yaml = write_data_yaml(plates_merged, ["plate"])
print(f"  Primary (indian-number-plate):     {n1} images")
print(f"  Supplement (indian-cars-number-plate): {n2} images")
print(f"  Total merged: {n1 + n2} images")
print(f"  Classes: ['plate']")
print(f"  YAML: {plates_yaml}\n")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# MERGE 2: Helmet Violation Detection
# motorcycle-helmet-detection: remap to binary, drop 'invalid'
# two-wheeler-helmet: already binary
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print("=" * 55)
print("MERGING: Helmet Violation Datasets")
print("=" * 55)

helmets_merged = DATASETS_DIR / "helmets_merged"
if helmets_merged.exists():
    shutil.rmtree(helmets_merged)

# Read primary dataset class names to build remap
primary_classes = load_class_names(DATASETS_DIR / "helmet_primary")
print(f"  Primary classes: {primary_classes}")

# Build remap: with_helmet=0, without_helmet=1
# full-faced, half-faced → 0 (with_helmet)
# no helmet, no-helmet, No Helmet → 1 (without_helmet)
# invalid → drop
helmet_remap = {}
helmet_drop = set()
for idx, name in enumerate(primary_classes):
    low = name.lower().strip()
    if low in ("full-faced", "half-faced", "half face", "full face"):
        helmet_remap[idx] = 0  # with_helmet
    elif "no" in low and "helmet" in low:
        helmet_remap[idx] = 1  # without_helmet
    elif low == "invalid":
        helmet_drop.add(idx)
    else:
        # Unknown — try to classify
        if "helmet" in low:
            helmet_remap[idx] = 0
        else:
            helmet_drop.add(idx)

print(f"  Remap: {helmet_remap}")
print(f"  Drop classes (indices): {helmet_drop}")

n3 = copy_split(DATASETS_DIR / "helmet_primary", helmets_merged, "mhd",
                class_remap=helmet_remap, drop_classes=helmet_drop)

# Supplement: read its classes and remap to match target
supp_classes = load_class_names(DATASETS_DIR / "helmet_supplement")
print(f"  Supplement classes: {supp_classes}")

supp_remap = {}
for idx, name in enumerate(supp_classes):
    low = name.lower().strip()
    if "without" in low or ("no" in low and "helmet" in low):
        supp_remap[idx] = 1  # without_helmet
    else:
        supp_remap[idx] = 0  # with_helmet

print(f"  Supplement remap: {supp_remap}")
n4 = copy_split(DATASETS_DIR / "helmet_supplement", helmets_merged, "twh",
                class_remap=supp_remap)

helmets_yaml = write_data_yaml(helmets_merged, ["with_helmet", "without_helmet"])
print(f"\n  Primary (motorcycle-helmet-detection): {n3} images")
print(f"  Supplement (two-wheeler-helmet):       {n4} images")
print(f"  Total merged: {n3 + n4} images")
print(f"  Classes: ['with_helmet', 'without_helmet']")
print(f"  YAML: {helmets_yaml}\n")

# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
# Vehicle dataset — no merge needed
# ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
print("=" * 55)
print("Vehicle Type Dataset (no merge needed)")
print("=" * 55)
vehicle_classes = load_class_names(DATASETS_DIR / "vehicles")
print(f"  Classes: {vehicle_classes}")
print(f"  YAML: {DATASETS_DIR / 'vehicles' / 'data.yaml'}")

print("\n\u2713 All datasets merged and ready for training!")

## 3. Model Training Configuration
**Training hyperparameters optimized for V100:**
| Parameter | Value | Rationale |
|-----------|-------|-----------|
| Model | YOLOv8s | 11.2M params, best speed/accuracy for V100 |
| Image Size | 640 | Standard YOLO input, fits V100 memory |
| Batch Size | 64 (plates & helmets), 32 (vehicles) | V100 32GB handles 64 for larger datasets |
| Epochs | 100 (plates & helmets), 150 (vehicles) | More epochs for smaller vehicle dataset |
| Patience | 15 (plates & helmets), 20 (vehicles) | Early stopping to prevent overfitting |
| Optimizer | AdamW | Better generalization than SGD |
| LR | 0.001 → 0.01 cosine | Default YOLOv8, cosine LR schedule |
| Augmentation | Default | Mosaic, HSV, flip, scale built into YOLO |

In [ ]:
# ══════════════════════════════════════════════════════════
# 3A. Train Model 1: Indian License Plate Detection
# ══════════════════════════════════════════════════════════
from ultralytics import YOLO

print("=" * 60)
print("TRAINING: Indian License Plate Detection (~2,100 images)")
print("=" * 60)

plate_model = YOLO("yolov8s.pt")  # Start from pretrained COCO weights

plate_results = plate_model.train(
    data=str(DATASETS_DIR / "plates_merged" / "data.yaml"),
    epochs=100,
    imgsz=640,
    batch=64,
    patience=15,
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    warmup_epochs=5,
    warmup_momentum=0.5,
    cos_lr=True,
    device=0,
    workers=8,
    project=str(MODELS_DIR),
    name="plate_detector",
    exist_ok=True,
    verbose=True,
    save=True,
    save_period=10,
    plots=True,
)

plate_best = MODELS_DIR / "plate_detector" / "weights" / "best.pt"
print(f"\n\u2713 Plate detection model saved: {plate_best}")
print(f"  mAP50: {plate_results.results_dict.get('metrics/mAP50(B)', 'N/A')}")
print(f"  mAP50-95: {plate_results.results_dict.get('metrics/mAP50-95(B)', 'N/A')}")

In [ ]:
# ══════════════════════════════════════════════════════════
# 3B. Train Model 2: Helmet Violation Detection
# ══════════════════════════════════════════════════════════
print("=" * 60)
print("TRAINING: Helmet Violation Detection (~2,520 images)")
print("  Classes: with_helmet / without_helmet")
print("=" * 60)

helmet_model = YOLO("yolov8s.pt")

helmet_results = helmet_model.train(
    data=str(DATASETS_DIR / "helmets_merged" / "data.yaml"),
    epochs=100,
    imgsz=640,
    batch=64,
    patience=15,
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    warmup_epochs=5,
    warmup_momentum=0.5,
    cos_lr=True,
    device=0,
    workers=8,
    project=str(MODELS_DIR),
    name="helmet_detector",
    exist_ok=True,
    verbose=True,
    save=True,
    save_period=10,
    plots=True,
)

helmet_best = MODELS_DIR / "helmet_detector" / "weights" / "best.pt"
print(f"\n\u2713 Helmet violation model saved: {helmet_best}")
print(f"  mAP50: {helmet_results.results_dict.get('metrics/mAP50(B)', 'N/A')}")
print(f"  mAP50-95: {helmet_results.results_dict.get('metrics/mAP50-95(B)', 'N/A')}")

In [ ]:
# ══════════════════════════════════════════════════════════
# 3C. Train Model 3: Vehicle Type Detection
# ══════════════════════════════════════════════════════════
print("=" * 60)
print("TRAINING: Vehicle Type Detection Model")
print("=" * 60)

vehicle_model = YOLO("yolov8s.pt")

# Smaller dataset (627 images) — use batch=32 and more augmentation epochs
vehicle_results = vehicle_model.train(
    data=str(DATASETS_DIR / "vehicles" / "data.yaml"),
    epochs=150,
    imgsz=640,
    batch=32,
    patience=20,
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.01,
    warmup_epochs=5,
    warmup_momentum=0.5,
    cos_lr=True,
    device=0,
    workers=8,
    project=str(MODELS_DIR),
    name="vehicle_classifier",
    exist_ok=True,
    verbose=True,
    save=True,
    save_period=10,
    plots=True,
)

vehicle_best = MODELS_DIR / "vehicle_classifier" / "weights" / "best.pt"
print(f"\n\u2713 Vehicle type model saved: {vehicle_best}")
print(f"  mAP50: {vehicle_results.results_dict.get('metrics/mAP50(B)', 'N/A')}")
print(f"  mAP50-95: {vehicle_results.results_dict.get('metrics/mAP50-95(B)', 'N/A')}")

## 4. Model Evaluation & Validation
Run inference on validation sets and generate confusion matrices, PR curves, and sample predictions.

In [ ]:
# ══════════════════════════════════════════════════════════
# 4. Evaluate All Models
# ══════════════════════════════════════════════════════════
from ultralytics import YOLO

models = {
    "Plate Detector": str(MODELS_DIR / "plate_detector" / "weights" / "best.pt"),
    "Helmet Detector": str(MODELS_DIR / "helmet_detector" / "weights" / "best.pt"),
    "Vehicle Classifier": str(MODELS_DIR / "vehicle_classifier" / "weights" / "best.pt"),
}

datasets = {
    "Plate Detector": str(DATASETS_DIR / "plates_merged" / "data.yaml"),
    "Helmet Detector": str(DATASETS_DIR / "helmets_merged" / "data.yaml"),
    "Vehicle Classifier": str(DATASETS_DIR / "vehicles" / "data.yaml"),
}

print("=" * 60)
print("MODEL EVALUATION RESULTS")
print("=" * 60)

sep = "\u2500" * 50
for name, model_path in models.items():
    print(f"\n{sep}")
    print(f"  {name}")
    print(f"{sep}")
    model = YOLO(model_path)
    results = model.val(data=datasets[name], imgsz=640, batch=32, device=0)
    print(f"  Precision:  {results.results_dict.get('metrics/precision(B)', 'N/A'):.4f}")
    print(f"  Recall:     {results.results_dict.get('metrics/recall(B)', 'N/A'):.4f}")
    print(f"  mAP@50:     {results.results_dict.get('metrics/mAP50(B)', 'N/A'):.4f}")
    print(f"  mAP@50-95:  {results.results_dict.get('metrics/mAP50-95(B)', 'N/A'):.4f}")

print(f"\n{'=' * 60}")
print("All models evaluated. Check runs/ for confusion matrices & PR curves.")
print(f"{'=' * 60}")

## 5. Export Models for Deployment
Export trained models to ONNX format for faster inference and deploy to the SVIES project.

In [ ]:
# ══════════════════════════════════════════════════════════
# 5. Export Models for Production Deployment
# ══════════════════════════════════════════════════════════
from ultralytics import YOLO

export_dir = PROJECT_DIR / "exported_models"
export_dir.mkdir(exist_ok=True)

for name, model_path in models.items():
    print(f"\nExporting {name}...")
    model = YOLO(model_path)
    
    # Export to ONNX (cross-platform, fast inference)
    onnx_path = model.export(format="onnx", imgsz=640, simplify=True)
    print(f"  \u2713 ONNX: {onnx_path}")
    
    # Export to TorchScript (for PyTorch deployment)
    ts_path = model.export(format="torchscript", imgsz=640)
    print(f"  \u2713 TorchScript: {ts_path}")

print(f"\n{'=' * 60}")
print("All models exported!")
print(f"{'=' * 60}")
print(f"\nTo use in SVIES project:")
print(f"  1. Download the best.pt files from: {MODELS_DIR}")
print(f"  2. Place them in your project's models/ directory:")
print(f"     - models/plate_detector.pt")
print(f"     - models/helmet_detector.pt")
print(f"     - models/vehicle_classifier.pt")
print(f"  3. Update config.py with the new model paths")

## 6. Test Inference on Sample Images
Quick visual verification that models work correctly before downloading.

In [ ]:
# ══════════════════════════════════════════════════════════
# 6. Test Inference on Sample Images
# ══════════════════════════════════════════════════════════
import matplotlib.pyplot as plt
from ultralytics import YOLO
import cv2
import glob

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle("SVIES Model Inference Results", fontsize=16, fontweight="bold")

for idx, (name, model_path) in enumerate(models.items()):
    model = YOLO(model_path)
    
    # Find a sample image from the validation set
    dataset_name = list(datasets.keys())[idx]
    data_yaml = datasets[dataset_name]
    data_dir = Path(data_yaml).parent
    
    val_images = list((data_dir / "valid" / "images").glob("*.*"))
    if not val_images:
        val_images = list((data_dir / "test" / "images").glob("*.*"))
    
    if val_images:
        sample = str(val_images[0])
        results = model(sample, imgsz=640)
        annotated = results[0].plot()
        annotated_rgb = cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB)
        axes[idx].imshow(annotated_rgb)
        axes[idx].set_title(name, fontsize=12)
    else:
        axes[idx].text(0.5, 0.5, "No validation images", ha="center", va="center")
        axes[idx].set_title(name, fontsize=12)
    
    axes[idx].axis("off")

plt.tight_layout()
plt.savefig(str(PROJECT_DIR / "inference_results.png"), dpi=150, bbox_inches="tight")
plt.show()
print("\u2713 Inference results saved to inference_results.png")

## 7. Training Summary & Next Steps
After training completes:
1. **Download models**: Copy `best.pt` files from `svies_training/models/` to your local `models/` directory
2. **Rename** for SVIES project:
   - `plate_detector/weights/best.pt` → `models/svies_plate_detector.pt`
   - `helmet_detector/weights/best.pt` → `models/svies_helmet_detector.pt`
   - `vehicle_classifier/weights/best.pt` → `models/yolov8n.pt` (or update config)
3. **Test locally**: Run `python main.py --source test_image.jpg` to verify
4. **Deploy**: Models are ready for the FastAPI backend

### Datasets Used (5 → merged into 3):
| Model | Datasets Merged | Total Images | Classes |
|-------|----------------|--------------|---------|
| Plate Detector | `indian-number-plate` + `indian-cars-number-plate` | ~2,100 | plate |
| Helmet Detector | `motorcycle-helmet-detection` + `two-wheeler-helmet` | ~2,520 | with_helmet, without_helmet |
| Vehicle Classifier | `vehicles-openimages` | 627 | Car, Bus, Motorcycle, Truck, Ambulance |

### Expected Performance (V100 trained):
| Model | Expected mAP@50 | Inference Time |
|-------|-----------------|----------------|
| Indian Plate Detector | 88-95% | ~8ms/image |
| Helmet Violation Detector | 82-90% | ~8ms/image |
| Vehicle Classifier | 70-82% | ~8ms/image |

**Total training time on V100: ~3-5 hours for all 3 models**